# 第 10 章：奇異值分解（SVD）

本 Notebook 是 [`ch10_svd.md`](ch10_svd.md) 的精簡對照版本，程式碼邏輯與 [`ch10_svd.py`](ch10_svd.py) 一致，可互動執行。

## 學習目標

- 寫出任意 $m \times n$ 矩陣的奇異值分解 $A = U \Sigma V^T$
- 說明 SVD 與特徵值分解的關係：奇異值是 $A^TA$（或 $AA^T$）特徵值的平方根
- 理解 SVD 的幾何意義：「旋轉 → 縮放 → 旋轉」，單位圓如何映射成橢圓
- 用奇異值個數判斷矩陣的秩（rank）
- 用 SVD 計算 Moore-Penrose 偽逆，並理解其在最小二乘解中的用途

In [1]:
import os

import matplotlib
matplotlib.use("Agg")  # 無顯示器環境，僅存檔不顯示

import matplotlib.pyplot as plt
import numpy as np

matplotlib.rcParams["font.sans-serif"] = [
    "PingFang TC", "PingFang SC", "Heiti TC", "Arial Unicode MS", "DejaVu Sans",
]
matplotlib.rcParams["axes.unicode_minus"] = False

OUT_DIR = os.path.abspath(".")
np.set_printoptions(precision=4, suppress=True)

## 1. SVD 的定義：$A = U \Sigma V^T$

任意 $m\times n$ 矩陣都能分解成 $U$（$m\times m$ 正交矩陣）、$\Sigma$（$m\times n$ 對角矩陣，對角線為由大到小排列的奇異值）、$V$（$n\times n$ 正交矩陣）。

In [2]:
A = np.array([
    [3.0, 0.0],
    [4.0, 5.0],
    [0.0, 0.0],
])
print("矩陣 A (3x2) =")
print(A)

U, s, Vt = np.linalg.svd(A, full_matrices=True)
print("\nU (3x3) =")
print(U)
print("\n奇異值 s =", s)
print("\nV^T (2x2) =")
print(Vt)

m, n = A.shape
Sigma = np.zeros((m, n))
Sigma[: len(s), : len(s)] = np.diag(s)
print("\nSigma (3x2) =")
print(Sigma)

矩陣 A (3x2) =
[[3. 0.]
 [4. 5.]
 [0. 0.]]

U (3x3) =
[[-0.3162 -0.9487  0.    ]
 [-0.9487  0.3162  0.    ]
 [ 0.      0.      1.    ]]

奇異值 s = [6.7082 2.2361]

V^T (2x2) =
[[-0.7071 -0.7071]
 [-0.7071  0.7071]]

Sigma (3x2) =
[[6.7082 0.    ]
 [0.     2.2361]
 [0.     0.    ]]


In [3]:
A_reconstructed = U @ Sigma @ Vt
print("重建 U @ Sigma @ V^T =")
print(A_reconstructed)
print("驗證 U @ Sigma @ V^T ≈ A ?", np.allclose(A_reconstructed, A))
assert np.allclose(A_reconstructed, A)

V = Vt.T
print("驗證 U 為正交矩陣 (U^T U ≈ I) ?", np.allclose(U.T @ U, np.eye(m)))
print("驗證 V 為正交矩陣 (V^T V ≈ I) ?", np.allclose(V.T @ V, np.eye(n)))

重建 U @ Sigma @ V^T =
[[3. 0.]
 [4. 5.]
 [0. 0.]]
驗證 U @ Sigma @ V^T ≈ A ? True
驗證 U 為正交矩陣 (U^T U ≈ I) ? True
驗證 V 為正交矩陣 (V^T V ≈ I) ? True


## 2. SVD 與特徵值分解的關係

$A^TA = V(\Sigma^T\Sigma)V^T$、$AA^T = U(\Sigma\Sigma^T)U^T$，所以：
- $V$ 的欄是 $A^TA$ 的特徵向量，$\sigma_i^2$ 是 $A^TA$ 的特徵值
- $U$ 的欄是 $AA^T$ 的特徵向量

In [4]:
AtA = A.T @ A
eigvals_AtA, eigvecs_AtA = np.linalg.eigh(AtA)
order = np.argsort(eigvals_AtA)[::-1]
eigvals_AtA = eigvals_AtA[order]
eigvecs_AtA = eigvecs_AtA[:, order]

print("A^T A 的特徵值 (由大到小) =", eigvals_AtA)
print("奇異值的平方 s^2 =", s ** 2)
print("驗證：特徵值 ≈ 奇異值平方 ?", np.allclose(eigvals_AtA, s ** 2))

print("\n兩者方向是否一致（取絕對值比較）？",
      np.allclose(np.abs(V[:, : len(s)]), np.abs(eigvecs_AtA[:, : len(s)])))

A^T A 的特徵值 (由大到小) = [45.  5.]
奇異值的平方 s^2 = [45.  5.]
驗證：特徵值 ≈ 奇異值平方 ? True

兩者方向是否一致（取絕對值比較）？ True


## 3. 手算範例：$2\times2$ 矩陣的完整 SVD

設 $B=\begin{bmatrix}1&1\\0&1\end{bmatrix}$。先求 $B^TB$ 的特徵值 $\lambda = \dfrac{3\pm\sqrt5}{2}$，再開根號得到奇異值。詳細手算過程請見 [`ch10_svd.md`](ch10_svd.md) 第 7 節。

In [5]:
B = np.array([
    [1.0, 1.0],
    [0.0, 1.0],
])
print("矩陣 B =")
print(B)

BtB = B.T @ B
eigvals_B, eigvecs_B = np.linalg.eigh(BtB)
order_B = np.argsort(eigvals_B)[::-1]
eigvals_B = eigvals_B[order_B]
singular_values_B = np.sqrt(eigvals_B)

print("\nB^T B 的特徵值 =", eigvals_B)
print("手算奇異值 = sqrt(特徵值) =", singular_values_B)

U_B, s_B, Vt_B = np.linalg.svd(B)
print("\nnumpy 計算的奇異值 =", s_B)
print("驗證一致？", np.allclose(singular_values_B, s_B))

矩陣 B =
[[1. 1.]
 [0. 1.]]

B^T B 的特徵值 = [2.618 0.382]
手算奇異值 = sqrt(特徵值) = [1.618 0.618]

numpy 計算的奇異值 = [1.618 0.618]
驗證一致？ True


## 4. 用 SVD 計算矩陣的秩

矩陣的秩等於非零奇異值的個數。

In [6]:
C = np.array([
    [1.0, 2.0, 3.0],
    [2.0, 4.0, 6.0],
    [1.0, 0.0, 1.0],
])
print("矩陣 C =")
print(C)

_, s_C, _ = np.linalg.svd(C)
print("\n奇異值 s =", s_C)

tol = 1e-10
rank_from_svd = int(np.sum(s_C > tol))
rank_numpy = np.linalg.matrix_rank(C)

print("非零奇異值個數 =", rank_from_svd)
print("np.linalg.matrix_rank(C) =", rank_numpy)
print("是否一致？", rank_from_svd == rank_numpy)

矩陣 C =
[[1. 2. 3.]
 [2. 4. 6.]
 [1. 0. 1.]]

奇異值 s = [8.4354 0.9183 0.    ]
非零奇異值個數 = 2
np.linalg.matrix_rank(C) = 2
是否一致？ True


## 5. Moore-Penrose 偽逆（Pseudo-Inverse）

$A^+ = V\Sigma^+U^T$，其中 $\Sigma^+$ 是把非零奇異值取倒數後轉置。$x^*=A^+b$ 給出 $Ax=b$ 的最小二乘解（第 11 章詳細介紹）。

In [7]:
Sigma_pinv = np.zeros((n, m))
for i in range(len(s)):
    if s[i] > tol:
        Sigma_pinv[i, i] = 1.0 / s[i]

A_pinv_manual = V @ Sigma_pinv @ U.T
A_pinv_numpy = np.linalg.pinv(A)

print("手算偽逆 A^+ =")
print(A_pinv_manual)
print("\nnp.linalg.pinv(A) =")
print(A_pinv_numpy)
print("\n驗證一致？", np.allclose(A_pinv_manual, A_pinv_numpy))

手算偽逆 A^+ =
[[ 0.3333 -0.      0.    ]
 [-0.2667  0.2     0.    ]]

np.linalg.pinv(A) =
[[ 0.3333 -0.      0.    ]
 [-0.2667  0.2     0.    ]]

驗證一致？ True


In [8]:
b = np.array([1.0, 2.0, 3.0])
x_lstsq = A_pinv_numpy @ b
x_check, *_ = np.linalg.lstsq(A, b, rcond=None)

print("最小二乘解 x = A^+ b =", x_lstsq)
print("np.linalg.lstsq(A, b) 的解 =", x_check)
print("是否一致？", np.allclose(x_lstsq, x_check))
print("\n驗證 A A^+ A ≈ A ?", np.allclose(A @ A_pinv_numpy @ A, A))

最小二乘解 x = A^+ b = [0.3333 0.1333]
np.linalg.lstsq(A, b) 的解 = [0.3333 0.1333]
是否一致？ True

驗證 A A^+ A ≈ A ? True


## 6. 幾何意義：單位圓映射成橢圓

$2\times2$ 矩陣 $M$ 把單位圓映射成橢圓：橢圓主軸方向為 $U$ 的欄，主軸半長為奇異值 $\sigma_i$。

In [9]:
M = np.array([
    [3.0, 1.0],
    [1.0, 2.0],
])
U_M, s_M, Vt_M = np.linalg.svd(M)
V_M = Vt_M.T
print("矩陣 M =")
print(M)
print("\n奇異值 s =", s_M)

theta = np.linspace(0, 2 * np.pi, 200)
circle = np.vstack([np.cos(theta), np.sin(theta)])
ellipse = M @ circle

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

ax = axes[0]
ax.plot(circle[0], circle[1], color="tab:blue", label="單位圓")
for i in range(2):
    vec = V_M[:, i]
    ax.quiver(0, 0, vec[0], vec[1], angles="xy", scale_units="xy", scale=1,
               color=f"C{i+1}", label=f"v{i+1} 方向 (V 第 {i+1} 欄)")
ax.set_xlim(-2, 2); ax.set_ylim(-2, 2); ax.set_aspect("equal")
ax.axhline(0, color="gray", linewidth=0.5); ax.axvline(0, color="gray", linewidth=0.5)
ax.grid(True, linestyle="--", alpha=0.5)
ax.set_title("映射前：單位圓與 V 的正交方向")
ax.legend(loc="upper left", fontsize=9)

ax = axes[1]
ax.plot(ellipse[0], ellipse[1], color="tab:blue", label="M 映射後的橢圓")
for i in range(2):
    vec = U_M[:, i] * s_M[i]
    ax.quiver(0, 0, vec[0], vec[1], angles="xy", scale_units="xy", scale=1,
               color=f"C{i+1}", label=f"sigma{i+1} * u{i+1}（奇異值={s_M[i]:.3f}）")
ax.set_xlim(-4, 4); ax.set_ylim(-4, 4); ax.set_aspect("equal")
ax.axhline(0, color="gray", linewidth=0.5); ax.axvline(0, color="gray", linewidth=0.5)
ax.grid(True, linestyle="--", alpha=0.5)
ax.set_title("映射後：橢圓與主軸（長度 = 奇異值）")
ax.legend(loc="upper left", fontsize=9)

fig.suptitle("SVD 的幾何意義：旋轉 -> 縮放 -> 旋轉（單位圓變橢圓）")
fig_path = os.path.join(OUT_DIR, "svd_circle_to_ellipse.png")
plt.savefig(fig_path, dpi=120, bbox_inches="tight")
plt.show()
print("已儲存示意圖至:", fig_path)

矩陣 M =
[[3. 1.]
 [1. 2.]]

奇異值 s = [3.618 1.382]


已儲存示意圖至: /Users/rexwang/workspace/linear-algebra-with-matlab-python-tutorial/ch10_svd/svd_circle_to_ellipse.png


/var/folders/3r/mbt455ns6n3fn8ythcyc95qc0000gn/T/ipykernel_53075/3113083405.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 重點整理

- $A = U\Sigma V^T$：$U$、$V$ 為正交矩陣，$\Sigma$ 對角線為由大到小排列的奇異值。
- 奇異值 $\sigma_i=\sqrt{\lambda_i(A^TA)}$；$V$ 的欄是 $A^TA$ 的特徵向量，$U$ 的欄是 $AA^T$ 的特徵向量。
- 幾何上是「旋轉 → 縮放 → 旋轉」：單位圓變橢圓，主軸方向為 $U$、半長為奇異值。
- 秩 = 非零奇異值個數。
- 偽逆 $A^+=V\Sigma^+U^T$ 可推廣反矩陣到任意矩陣，$x^*=A^+b$ 給出最小二乘解，是第 11 章的理論基礎。